# 5. PDF Onboarding

**Goal:** Enable the agent to answer questions grounded in unstructured PDF documents.

**Key Concept:**
We use a **Vector Database (VDB)** to chunk and embed a PDF (ERCOT market briefing), enabling a standard **RAG (Retrieval Augmented Generation)** workflow: retrieve relevant excerpts first, then answer with citations grounded in the document.

This notebook shows two paths:
1. Use an **already deployed** VDB via MCP.
2. **Create + deploy** a new VDB using the DataRobot Python SDK, then query it via MCP.

## Option 1: Use an already deployed Vector Database (fast path)

**Goal:** Connect to an existing VDB deployment via MCP and ask a question.

**Key Concept:**
Your agent does not need to re-index the PDF if a VDB deployment already exists; it can query the VDB tool directly via MCP.

In [ ]:
import os
from pprint import pprint
from dotenv import load_dotenv
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

# 1. Load configuration
load_dotenv(override=True)

# 2. Initialize client
dr_client = dr.Client()

# 3. Configure the tool connection (MCP)
# This connects to a deployment that exposes the VDB retrieval tool.
MCP_DEPLOYMENT_ID = os.getenv("MCP_DEPLOYMENT_ID")
if not MCP_DEPLOYMENT_ID:
    raise ValueError("MCP_DEPLOYMENT_ID environment variable is not set")
print(f"Using MCP_DEPLOYMENT_ID={MCP_DEPLOYMENT_ID}")

server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    max_retries=3,
    timeout=60.0,
)

# 4. Configure the model (via DataRobot LLM Gateway)
MODEL_NAME = os.getenv("MODEL_NAME")
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=dr_client.endpoint + "/genai/llmgw",
    ),
)

# 5. Define the agent
system_prompt = """
You are a Compliance Assistant for the ERCOT Market Briefing.
Use the available ERCOT Vector Database to verify ERCOT Key Metrics.
""".strip()

agent = Agent(model=model, toolsets=[server], system_prompt=system_prompt)

# 6. Execution (example question)
question = "How many customers lost power during Hurricane Francine?"
print(f"\nQuestion: {question}")
print("-" * 30)

async with server:
    response = await agent.run(question)
    pprint(response.output)

## Option 2: Create and deploy a new Vector Database (full setup)

**Goal:** Upload the PDF, create a VDB (chunking + embeddings), send it to the Custom Model Workshop, register it, deploy it, then query it via MCP.

**Key Concept:**
A VDB is treated like a deployable asset in DataRobot. Once deployed, it can be used as an MCP tool by an agent for RAG-style Q&A.

**Before you run Option 2:**

- Set `VECTOR_DB_NAME` in `.env` to a unique value for your run.

This section is organized as a single pipeline:

- **Step 1 — Upload**: Upload the PDF to the AI Catalog (as a ZIP).
- **Step 2 — Create**: Create the VectorDB (chunking + embeddings) and wait for completion.
- **Step 3 — Deploy + Query**: Deploy the VDB and query it via MCP from an agent.

### Step 1: Upload the PDF to the AI Catalog (as a ZIP)

**Why ZIP?** DataRobot’s AI Catalog upload expects an archive for PDFs in this workflow.

This step will:
- Create a ZIP containing the PDF
- Upload it to the AI Catalog
- Capture the uploaded `dataset.id` for VDB creation

In [ ]:
import os
from dotenv import load_dotenv

# Name of the Vector Database (set VDB_NAME in .env)
load_dotenv(override=True)
vDB_name = os.getenv("VECTOR_DB_NAME")
print(f"Using VECTOR_DB_NAME={vDB_name}")

In [ ]:
import datarobot as dr
from datarobot.models.genai.vector_database import VectorDatabase
from datarobot.models.genai.vector_database import ChunkingParameters
from datarobot.enums import VectorDatabaseEmbeddingModel
from datarobot.enums import VectorDatabaseChunkingMethod
from datarobot.enums import PredictionEnvironmentPlatform
from datarobot.enums import PredictionEnvironmentModelFormats
import time
import requests

In [ ]:
# 1. Connect to DataRobot
# Uses your notebook/session credentials.
dr.Client()

# 2. Zip the PDF (required for upload in this workflow)
import os
import zipfile
import tempfile

pdf_path = "documents/ercot_market_briefing_enhanced.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found at: {pdf_path}")

print(f"Creating ZIP file from {pdf_path}...")
with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp_zip:
    zip_path = tmp_zip.name
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(pdf_path, os.path.basename(pdf_path))
    print(f"ZIP file created: {zip_path}")

# 3. Upload ZIP to the AI Catalog
try:
    print("Uploading ZIP file to DataRobot AI Catalog...")
    dataset = dr.Dataset.create_from_file(file_path=zip_path)
    print(f"Dataset uploaded successfully. Dataset ID: {dataset.id}")
finally:
    # 4. Clean up local temp ZIP
    if os.path.exists(zip_path):
        os.remove(zip_path)
        print(f"Cleaned up temporary ZIP file: {zip_path}")

In [ ]:
# 1. Define chunking + embedding parameters
# These settings control how the PDF is split into chunks and embedded.
chunking_parameters = ChunkingParameters(
    embedding_model=VectorDatabaseEmbeddingModel.JINA_EMBEDDING_T_EN_V1,
    chunking_method=VectorDatabaseChunkingMethod.RECURSIVE,
    chunk_size=256,
    chunk_overlap_percentage=25,
    separators=["\n\n", "\n", " ", ""],
)

### Step 2: Create the Vector Database (chunking + embeddings)

In this step we:
- Define chunking/embedding parameters
- Create the VDB from the uploaded PDF dataset
- Wait until the VDB build completes

In [ ]:
# 2. Create the Vector Database
# Note: use_case is optional but recommended for organization/governance.
vdb = VectorDatabase.create(
    dataset_id=dataset.id,  # uploaded PDF dataset
    chunking_parameters=chunking_parameters,
    use_case=os.environ.get("DATAROBOT_DEFAULT_USE_CASE"),
    name=vDB_name,
)

print(f"VDB created: {vdb.id} ({getattr(vdb, 'name', 'unnamed')})")

In [ ]:
# 3. Wait for VDB creation to complete
max_wait_time = 600
check_interval = 5
start_time = time.time()

print("Waiting for vector database creation...")
while time.time() - start_time < max_wait_time:
    vdb = VectorDatabase.get(vdb.id)
    status = vdb.execution_status

    if status == "COMPLETED":
        print(f"✅ Vector database created: {vdb.name}")
        break

    if status == "FAILED":
        error_msg = getattr(vdb, "error_message", "Unknown error")
        raise RuntimeError(f"Vector database creation failed: {error_msg}")

    # Show progress if available
    percentage = getattr(vdb, "percentage", None)
    if percentage is not None:
        print(f"  Status: {status} ({percentage}%)")
    else:
        print(f"  Status: {status}...")

    time.sleep(check_interval)
else:
    raise TimeoutError(f"Vector database creation timed out after {max_wait_time} seconds")

assert (
    vdb.execution_status == "COMPLETED"
), f"Vector database creation failed with status: {vdb.execution_status}"

In [ ]:
# Step 2.5: Smoke test VDB build before deployment
vdb = VectorDatabase.get(vdb.id)
assert vdb.execution_status == "COMPLETED", (
    f"VDB must be COMPLETED. Current status: {vdb.execution_status}"
)

chunks_count = getattr(vdb, "chunks_count", 0) or 0
print(f"VDB status: {vdb.execution_status}")
print(f"Chunks count: {chunks_count}")

if chunks_count == 0:
    raise RuntimeError("VDB smoke test failed: 0 chunks were created.")

export_job = vdb.submit_export_dataset_job()
print(f"Export job submitted: {export_job.job_id}")
print(f"Export dataset id: {export_job.export_dataset_id}")

print("\n✅ VDB pre-deployment smoke test passed")

FYI: In our DataRobot SDK version, VectorDatabase does not expose a direct query() method, regardless of deployment state.

- Before deployment: you can build/inspect/export the VDB object.
- Retrieval/query happens through a served interface (e.g., deployed endpoint / MCP tool).

In [ ]:
# Step 3: Deploy the VDB (Prediction Environment + resource sizing)

# 3.1 Create or reuse a Prediction Environment
PREDICTION_ENVIRONMENT_NAME = "Vector Database Prediction Environment"

prediction_environment = None
for env in dr.PredictionEnvironment.list():
    if env.name == PREDICTION_ENVIRONMENT_NAME:
        prediction_environment = env
        break

if prediction_environment is None:
    prediction_environment = dr.PredictionEnvironment.create(
        name=PREDICTION_ENVIRONMENT_NAME,
        platform=PredictionEnvironmentPlatform.DATAROBOT_SERVERLESS,
        supported_model_formats=[
            PredictionEnvironmentModelFormats.DATAROBOT,
            PredictionEnvironmentModelFormats.CUSTOM_MODEL,
        ],
    )
    print(f"Created prediction environment: {prediction_environment.name}")
else:
    print(f"Using existing prediction environment: {prediction_environment.name}")

# 3.2 (Optional) Select a resource bundle (prefers 3XL if available)
resource_bundle_id = None
try:
    dr_client = dr.Client()
    bundles_url = f"{dr_client.endpoint}/mlops/compute/bundles/"
    headers = {"Authorization": f"Bearer {dr_client.token}"}
    bundles_response = requests.get(
        bundles_url,
        headers=headers,
        params={"useCases": "customModel"},
    )

    if bundles_response.status_code == 200:
        bundles_data = bundles_response.json()
        if bundles_data.get("data"):
            bundles = bundles_data["data"]
            bundle_3xl = next(
                (b for b in bundles if "3XL" in b.get("name", "").upper()),
                None,
            )
            if bundle_3xl:
                resource_bundle_id = bundle_3xl["id"]
                print(f"Selected 3XL bundle: {bundle_3xl['name']}")
            else:
                sorted_bundles = sorted(
                    bundles,
                    key=lambda b: b.get("memoryBytes", 0),
                    reverse=True,
                )
                if sorted_bundles:
                    resource_bundle_id = sorted_bundles[0]["id"]
                    print(
                        "Warning: 3XL bundle not found. Using largest available: "
                        f"{sorted_bundles[0]['name']}"
                    )
        else:
            print("Using memory settings (no resource bundles available)")
    else:
        print("Using memory settings (resource bundles not enabled)")
except (ImportError, KeyError) as e:
    print(f"Using memory settings (error checking bundles): {e}")
except requests.RequestException as e:
    print(f"Using memory settings (network error checking bundles): {e}")

In [ ]:
# Step 3.3 Send the VDB to the Custom Model Workshop (so it can be deployed)
assert (
    vdb.execution_status == "COMPLETED"
), f"Vector database must be completed. Current status: {vdb.execution_status}"

if resource_bundle_id:
    custom_model_version = vdb.send_to_custom_model_workshop(
        resource_bundle_id=resource_bundle_id,
        replicas=1,
        network_egress_policy=dr.NETWORK_EGRESS_POLICY.PUBLIC,
    )
else:
    # Fallback: explicit memory sizing if bundles aren't available in your tenant.
    custom_model_version = vdb.send_to_custom_model_workshop(
        maximum_memory=4096 * 1024 * 1024,
        replicas=1,
        network_egress_policy=dr.NETWORK_EGRESS_POLICY.PUBLIC,
    )

print(f"Custom model version created: {custom_model_version}")

In [ ]:
# Step 3.4 Register the VDB as a model (Model Registry)
REGISTERED_MODEL_NAME = f"Vector Database - {vdb.name}"

# Register model (adds a new version if the registered model already exists)
existing_models = [m for m in dr.RegisteredModel.list() if m.name == REGISTERED_MODEL_NAME]

if existing_models:
    registered_model_version = dr.RegisteredModelVersion.create_for_custom_model_version(
        custom_model_version_id=custom_model_version.id,
        registered_model_id=existing_models[0].id,
    )
    print(f"Added new version to existing registered model: {REGISTERED_MODEL_NAME}")
else:
    registered_model_version = dr.RegisteredModelVersion.create_for_custom_model_version(
        custom_model_version_id=custom_model_version.id,
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    print(f"Created new registered model: {REGISTERED_MODEL_NAME}")

print(f"Registered model version id: {registered_model_version.id}")


In [ ]:
# Step 3.5 Wait for the registered model build to be ready
registered_model = dr.RegisteredModel.get(registered_model_version.registered_model_id)
max_wait_time = 600
check_interval = 10
start_time = time.time()

print("Waiting for model build to complete...")
while time.time() - start_time < max_wait_time:
    version = registered_model.get_version(registered_model_version.id)
    build_status = getattr(version, "build_status", None) or getattr(version, "buildStatus", None)

    if build_status in ("READY", "complete", "COMPLETE"):
        print(f"Model build completed (status: {build_status})")
        break

    if build_status in ("FAILED", "ERROR", "error"):
        raise RuntimeError(f"Model build failed. Status: {build_status}")

    print(f"  Build status: {build_status}...")
    time.sleep(check_interval)
else:
    version = registered_model.get_version(registered_model_version.id)
    build_status = getattr(version, "build_status", None) or getattr(version, "buildStatus", None)
    raise TimeoutError(f"Model build timed out. Current status: {build_status}")

# Verify ready status
version = registered_model.get_version(registered_model_version.id)
final_status = getattr(version, "build_status", None) or getattr(version, "buildStatus", None)
if final_status not in ("READY", "complete", "COMPLETE"):
    raise RuntimeError(f"Model not ready for deployment. Status: {final_status}")


In [ ]:
# Step 3.6 Deploy the registered VDB model

deployment = dr.Deployment.create_from_registered_model_version(
    registered_model_version.id,
    label=f"Vector Database Deployment - {vdb.name}",
    description="Vector database deployment for RAG applications",
    prediction_environment_id=prediction_environment.id,
    max_wait=600,
)

print(f"Deployment created: {deployment.id}")
print("Set MCP_DEPLOYMENT_ID to this deployment id to query via MCP.")


### Step 3: Query the deployed VDB via MCP

Once the deployment is created, use its **Deployment ID** as `MCP_DEPLOYMENT_ID`.

- If you just created the deployment in this notebook, you can set `MCP_DEPLOYMENT_ID = deployment.id`.
- Otherwise, set `MCP_DEPLOYMENT_ID` in your `.env` or environment to the deployment you want to query.

In [0]:
import os
from pprint import pprint

import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

# Step 3.7 Query the deployed VDB via MCP
# Use the deployment you created above, or set MCP_DEPLOYMENT_ID in your environment.
MCP_DEPLOYMENT_ID = os.getenv("MCP_DEPLOYMENT_ID")
if not MCP_DEPLOYMENT_ID:
    raise ValueError("MCP_DEPLOYMENT_ID environment variable is not set")

# 1. Setup MCP
dr_client = dr.Client()
server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    max_retries=3,
    timeout=60.0,
)

# 2. Configure model (via DataRobot LLM Gateway)
MODEL_NAME = os.getenv("MODEL_NAME")
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=f"{dr_client.endpoint}/genai/llmgw",
    ),
)

# 3. Define agent
system_prompt = (
    "You are a Compliance Assistant for the ERCOT Market Briefing. "
    "Use the Vector Database deployment to verify ERCOT Key Metrics."
)
agent = Agent(model=model, toolsets=[server], system_prompt=system_prompt)

# 4. Run query
async with server:
    question = "How much economic curtailment occurred in September 2025?"
    print(f"\nQuestion: {question}")
    print("-" * 30)
    response = await agent.run(question)
    pprint(response.output)
